### ENGR 418 Applied Machine Learning for Engineers - Project stage 1 & 2

Description:
* A 4-class image classifier that categorizes the LEGO image as: 2x1, circle, rectangle, square
  
Student names: 
* Zayd Amin
* Kevin Puthanagadi

Date: 
* November 29, 2025


In [1]:
# === Imports and configuration ===

# Core libraries for plotting, arrays, filesystem, randomness
import matplotlib.pyplot as plt
import numpy as np
import os
import random

# Image and ML utilities
from PIL import Image, ImageFilter
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
import cv2 as cv

# Dataset paths (relative to notebook location)
FOLDER_TRAINING = 'Lego_dataset_2/training/'
FOLDER_TESTING = 'Lego_dataset_2/testing/'

# Image processing parameters (stage 1)
# Default width/height used when resizing images for feature extraction
DEFAULT_IMAGE_WIDTH = 64
# Extra padding used when cropping images (if cropping is enabled later)
CROP_WIDTH_BUFFER = 15
CROP_HEIGHT_BUFFER = 20
# Fraction to crop from each side if cropping is used
CROP_FRACTION = 0.25
####################

# Model parameters (not all used directly but kept for clarity)
REGULARIZATION_STRENGTH = 0.1  # C parameter for LogisticRegression (1/lambda)
SOLVER_TYPE = 'lbfgs'          # Solver for LogisticRegression
PENALTY_TYPE = 'l2'            # Regularization type

# Class configuration maps numeric labels to filename prefixes
CLASS_LABELS = {
    0: '2x1',  # 2x1 lego piece (file names start with '2b1')
    1: 'cir',  # Circle lego piece (file names start with 'cir')
    2: 'rec',  # Rectangle lego piece (file names start with 'rec')
    3: 'squ'   # Square lego piece (file names start with 'squ')
}
NUM_CLASSES = len(CLASS_LABELS)

# Display parameters for sample visualization (not used in current run)
SAMPLE_INDICES = [30, 60, 90, 120]
TEST_SAMPLE_INDICES = [17, 32, 53, 65]
ACCURACY_ROUND_DIGITS = 4

# Load data sets for training and testing
# IM_SIZE controls the target resolution for resizing all images (For stage 2)
IM_SIZE = 900
# ET retained from earlier experiments (edge threshold placeholder)
ET = 60


# STAGE 1 CODE

In [2]:
# === Stage 1 baseline: data loading and preprocessing ===
print("Loading training data with stage 1 pipeline...")

def get_data(folder_path, image_width, class_labels_dict):
    """
    Load and preprocess LEGO images from a specified folder.
    
    This function reads all images from the given folder, converts them to grayscale,
    crops them to focus on the central region, resizes them to a standard size,
    and flattens them into feature vectors for machine learning.
    
    Parameters
    ----------
    folder_path : str
        Path to the folder containing image files (training or testing set)
    image_width : int
        Target width and height in pixels for the processed square images
    class_labels_dict : dict
        Dictionary mapping class numbers (int) to filename prefixes (str)
        Example: {0: '2b1', 1: 'cir', 2: 'rec', 3: 'squ'}
    
    Returns
    -------
    tuple
        x_data : numpy.ndarray
            Array of shape (n_samples, image_width²) containing flattened image data
        y_data : numpy.ndarray
            Array of shape (n_samples, 1) containing class labels (0, 1, 2, or 3)
    
    Notes
    -----
    - Images are converted to grayscale for consistent processing
    - Central cropping removes background and focuses on the LEGO piece
    - All images are resized to the same dimensions for uniform feature vectors
    """
    # Get list of all files in the specified folder
    file_names = os.listdir(folder_path)
    total_files = len(file_names)
    print(f"Total files found in {folder_path}: {total_files}")
    
    # Initialize arrays to store image data and labels
    x_data = np.empty((total_files, image_width ** 2))
    y_data = np.empty((total_files, 1))
    
    current_index = 0  # Track current position in arrays
    
    # Process images for each class
    for class_number, label_prefix in class_labels_dict.items():
        # Filter files belonging to current class based on filename prefix
        class_files = [fname for fname in file_names if str(label_prefix) in fname]
        print(f"Class {class_number} ({label_prefix}): {len(class_files)} images")
        
        # Process each image in the current class
        for filename in class_files:
            image_path = os.path.join(folder_path, filename)
            
            # Load image and convert to grayscale
            image = Image.open(image_path).convert('L')
            
            # Calculate crop boundaries to focus on central region
            left_bound = int(np.floor(image.width * CROP_FRACTION)) - CROP_WIDTH_BUFFER
            top_bound = int(np.floor(image.height * CROP_FRACTION)) - CROP_HEIGHT_BUFFER
            right_bound = int(np.ceil(image.width * (1 - CROP_FRACTION))) + CROP_WIDTH_BUFFER
            bottom_bound = int(np.ceil(image.height * (1 - CROP_FRACTION))) + CROP_HEIGHT_BUFFER
            
            # Crop and resize image to standard dimensions
            processed_image = image.crop((left_bound, top_bound, right_bound, bottom_bound))
            processed_image = processed_image.resize((image_width, image_width))
            
            # Convert to numpy array and flatten to 1D feature vector
            image_array = np.asarray(processed_image)
            x_data[current_index, :] = image_array.reshape(1, -1)
            y_data[current_index, 0] = int(class_number)
            
            current_index += 1
    
    return x_data, y_data

x_train_stage1, y_train_stage1 = get_data(FOLDER_TRAINING, DEFAULT_IMAGE_WIDTH, CLASS_LABELS)
print(f"Stage 1 training data shape: X={x_train_stage1.shape}, y={y_train_stage1.shape}")


Loading training data with stage 1 pipeline...
Total files found in Lego_dataset_2/training/: 108
Class 0 (2x1): 27 images
Class 1 (cir): 27 images
Class 2 (rec): 27 images
Class 3 (squ): 27 images
Stage 1 training data shape: X=(108, 4096), y=(108, 1)


In [3]:
# === Stage 1 baseline: training and evaluation ===
print("Initializing and training the stage 1 logistic regression classifier...")

model_stage1 = LogisticRegression(
    solver=SOLVER_TYPE,
    penalty=PENALTY_TYPE,
    C=REGULARIZATION_STRENGTH
)
model_stage1.fit(x_train_stage1, y_train_stage1.ravel())
print("Stage 1 model training completed successfully!")

def evaluate_model_performance(x_data, y_true, model, dataset_name=""):
    """
    Evaluate model performance and display confusion matrix and accuracy.
    
    Parameters
    ----------
    x_data : numpy.ndarray
        Feature data for evaluation
    y_true : numpy.ndarray
        True class labels
    model : sklearn model
        Trained machine learning model
    dataset_name : str
        Name of the dataset being evaluated
    
    Returns
    -------
    tuple
        (y_predicted, accuracy_score)
    """
    # Generate predictions
    y_predicted = model.predict(x_data)

    # Calculate metrics
    conf_matrix = confusion_matrix(y_true, y_predicted)
    accuracy = accuracy_score(y_true, y_predicted)
    
    # Display results
    print(f"=== {dataset_name.upper()} SET PERFORMANCE ===")
    print(f"Confusion Matrix:")
    print(conf_matrix)
    print(f"\nAccuracy Score: {np.round(accuracy, ACCURACY_ROUND_DIGITS)}")
    print("-" * 50)
    
    return y_predicted, accuracy

# Evaluate training set performance
print("Evaluating stage 1 model performance on training data...")
y_train_pred_stage1, train_accuracy_stage1 = evaluate_model_performance(
    x_train_stage1, y_train_stage1.ravel(), model_stage1, "training"
)


Initializing and training the stage 1 logistic regression classifier...
Stage 1 model training completed successfully!
Evaluating stage 1 model performance on training data...
=== TRAINING SET PERFORMANCE ===
Confusion Matrix:
[[27  0  0  0]
 [ 0 27  0  0]
 [ 0  0 27  0]
 [ 0  0  0 27]]

Accuracy Score: 1.0
--------------------------------------------------


<local path>\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
# === Stage 1 baseline: testing ===

def test_function_stage1(folder_path, image_width, class_labels_dict, model):
    """
    Test the trained model on a dataset and display performance metrics.

    This function loads test data, makes predictions, and evaluates the model's
    performance using confusion matrix and accuracy score.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing test images
    image_width : int
        Width/height of processed square images
    class_labels_dict : dict
        Dictionary mapping class numbers to filename prefixes
    model : sklearn model
        Trained machine learning model to evaluate

    Returns
    -------
    tuple
        (x_test, y_test) - Test features and labels for further analysis
    """
    print("Loading test data (stage 1 pipeline)...")
    x_test_stage1, y_test_stage1 = get_data(folder_path, image_width, class_labels_dict)
    print(f"Stage 1 test data shape: X={x_test_stage1.shape}, y={y_test_stage1.shape}")
    y_test_pred_stage1, test_accuracy_stage1 = evaluate_model_performance(
        x_test_stage1, y_test_stage1.ravel(), model, "testing"
    )
    return x_test_stage1, y_test_stage1

x_test_stage1, y_test_stage1 = test_function_stage1(
    FOLDER_TESTING, DEFAULT_IMAGE_WIDTH, CLASS_LABELS, model_stage1
)


Loading test data (stage 1 pipeline)...
Total files found in Lego_dataset_2/testing/: 108
Class 0 (2x1): 27 images
Class 1 (cir): 27 images
Class 2 (rec): 27 images
Class 3 (squ): 27 images
Stage 1 test data shape: X=(108, 4096), y=(108, 1)
=== TESTING SET PERFORMANCE ===
Confusion Matrix:
[[ 8 10  6  3]
 [ 3 14  5  5]
 [ 1  2 16  8]
 [ 1 10  5 11]]

Accuracy Score: 0.4537
--------------------------------------------------


# STAGE 2 CODE

In [5]:
# === Feature engineering helpers ===

# area, height, width, perimeter, aspect ratio
def relative_sizes(image):
    """
    Extract shape-based size metrics from a LEGO image.

    This function normalizes lighting, creates a binary mask, finds the main
    contour, and returns geometry metrics that describe the piece.

    Parameters
    ----------
    image : numpy.ndarray
        BGR image array representing a single LEGO piece.

    Returns
    -------
    list
        [width, height, aspect_ratio, area, relative_area, perimeter] for the main contour.

    Notes
    -----
    - Uses brightness/contrast adjustment and median blur before thresholding
    - Morphological open/close cleans noise and fills holes
    - Relative area normalizes contour area by its bounding rectangle
    """
    # Adjust brightness/contrast to normalize lighting across images
    #   alpha and gamma values adjusted after tuning
    alpha_b = 0.92
    gamma_b = 20
    brightness_image = cv.addWeighted(image, alpha_b, image, 0, gamma_b)
    alpha_c = 2.063
    gamma_c = -135
    bright_contrast_img = cv.addWeighted(brightness_image, alpha_c, brightness_image, 0, gamma_c)

    # Convert to grayscale
    gray_image = cv.cvtColor(bright_contrast_img, cv.COLOR_BGR2GRAY)

    # apply median blur to remove noise and preserves edges
    #   kernel size set to 9 (must be odd number)
    filtered_image = cv.medianBlur(gray_image, 9)

    # Binary mask via Otsu to isolate the LEGO piece
    #   thresholding (convert to binary) - otsu method
    #   0 is an arbitrary threshold since otsu finds the optimal threshold
    _, threshold_image = cv.threshold(filtered_image, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)

    # Morphological open/close removes small noise and fills holes
    kernel = np.ones((3, 3), np.uint8)
    smoothen_image = cv.morphologyEx(threshold_image, cv.MORPH_OPEN, kernel)
    smoothen_image = cv.morphologyEx(smoothen_image, cv.MORPH_CLOSE, kernel)

    # Find outer contour representing the piece silhouette
    contours, _ = cv.findContours(smoothen_image, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError('No contours found in image')
    
    # pick most prominent contour
    cnt = max(contours, key=cv.contourArea)
    
    # 1) height and width size
    #   rotated rectangle (minAreaRect) used to consider rotated lego pieces
    #   Minimum-area rectangle gives rotation-agnostic width/height
    (_, size, _) = cv.minAreaRect(cnt)
    width, height = size

    # keeping the naming convention constant (width = longer side) - ensures consitent aspect ratio calculation
    if height > width:
        width, height = height, width

    # 2) aspect ratio
    aspect_ratio = width/height if height > 0 else 0
    
    # Area/perimeter plus relative area capture shape fullness
    
    # 3) area
    area = cv.contourArea(cnt)
    relative_area = area/(width*height) if width*height else 0  # area of contour normalised compared to area of bounding rectange - good to distinguish circle
    
    # 4) perimeter
    perimeter = cv.arcLength(cnt, True)
    
    return [width, height, aspect_ratio, area, relative_area, perimeter]

# Large circle detector
def large_circle_detector(image):
    """
    Detect presence of a large circular LEGO piece using Hough transform.

    This function enhances contrast, blurs to reduce noise, thresholds, and
    applies HoughCircles tuned for large round pieces.

    Parameters
    ----------
    image : numpy.ndarray
        BGR image array representing a single LEGO piece.

    Returns
    -------
    int
        Number of large circles detected (often 0 or 1).
    """
    # Normalize brightness/contrast before edge detection
    #   alpha and gamma values adjusted after tuning
    alpha_b = 0.92
    gamma_b = 20
    brightness_image = cv.addWeighted(image, alpha_b, image, 0, gamma_b)
    
    alpha_c = 2.063
    gamma_c = -135
    bright_contrast_img = cv.addWeighted(brightness_image, alpha_c, brightness_image, 0, gamma_c)
    
    # Convert image to grayscale
    gray_image = cv.cvtColor(bright_contrast_img, cv.COLOR_BGR2GRAY)
    
    # apply median blur to remove noise and preserves edges
    #   kernel size set to 9 (must be odd number)
    filtered_image = cv.medianBlur(gray_image, 9)
    
    # Binary mask via Otsu to isolate the LEGO piece
    #   thresholding (convert to binary) - otsu method
    #   0 is an arbitrary threshold since otsu finds the optimal threshold
    #   Threshold type THRESH_TOZERO_INV returns the lego piece in its orignal intensity (not white) - observed to be good for edge detection
    _, threshold_image_otsu = cv.threshold(filtered_image,0,255,cv.THRESH_TOZERO_INV + cv.THRESH_OTSU)
    
    # perform circular hough transform to detect large circle lego piece:--------
    """
    gray = Input image (grayscale).
    HOUGH_GRADIENT: detection method.
    dp = inverse ratio of the accumulator resolution to the image resolution (set to 1)
    min_dist = Minimum distance between detected centers.
    param_1 = Upper threshold for the internal Canny edge detector.
    param_2 = Threshold for center detection.
    min_radius = 80: Minimum radius of circle to be detected
    max_radius = 120: Maximum radius of circle to be detected.
    min and max radius values set high to detect large cirlce
    """
    circles = cv.HoughCircles(threshold_image_otsu, cv.HOUGH_GRADIENT, 1, minDist=200,
            param1=50, param2=25,
            minRadius=80, maxRadius=120)
    
    circle = len(circles[0, :]) if circles is not None else 0
    
    return circle

# Number of studs
def process_stud_num(image):
    """
    Count LEGO studs using circular Hough detection.

    This function thresholds a denoised grayscale image and applies HoughCircles
    configured for typical stud size and spacing.

    Parameters
    ----------
    image : numpy.ndarray
        BGR image array representing a single LEGO piece.

    Returns
    -------
    int
        Number of studs detected on the piece.
    """
    # Convert image to grayscale
    gray_image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)

    # apply median blur to remove noise and preserves edges
    #   kernel size set to 9 (must be odd number)   
    filtered_image = cv.medianBlur(gray_image, 9)

    # Binary mask via Otsu to isolate the LEGO piece
    #   thresholding (convert to binary) - otsu method
    #   0 is an arbitrary threshold since otsu finds the optimal threshold
    #   Threshold type THRESH_TOZERO_INV returns the lego piece in its orignal intensity (not white) - observed to be good for edge detection
    _, threshold_image_otsu = cv.threshold(filtered_image,0,255,cv.THRESH_TOZERO_INV + cv.THRESH_OTSU)
    
    # perform circular hough transform to detect studs on lego piece:--------
    """
    gray = Input image (grayscale).
    HOUGH_GRADIENT: detection method.
    dp = inverse ratio of the accumulator resolution to the image resolution (set to 1)
    min_dist = Minimum distance between detected centers.
    param_1 = Upper threshold for the internal Canny edge detector.
    param_2 = Threshold for center detection.
    min_radius = 29: Minimum radius of circle to be detected
    max_radius = 40: Maximum radius of circle to be detected.
    min and max radius values set low to detect studs
    param1 and param2 set low as the edges of the studs are thin
    """
    circles = cv.HoughCircles(threshold_image_otsu, cv.HOUGH_GRADIENT, 1, minDist=40,
            param1=15, param2=15,
            minRadius=29, maxRadius=40)
    
    num_studs = len(circles[0, :]) if circles is not None else 0
    
    return num_studs

# Hu moments (silhoute detection)
def process_hu_moments(image):
    """
    Compute Hu moment vector to capture shape invariants of a LEGO piece.

    This function normalizes brightness, segments the main contour, and returns
    log-transformed Hu moments for shape invariance.

    Parameters
    ----------
    image : numpy.ndarray
        BGR image array representing a single LEGO piece.

    Returns
    -------
    numpy.ndarray
        Array of shape (7,) containing log-transformed Hu moments.
    """
    # Normalize brightness/contrast before edge detection
    #   alpha and gamma values adjusted after tuning
    alpha_b = 0.92
    gamma_b = 20
    brightness_image = cv.addWeighted(image, alpha_b, image, 0, gamma_b)
    
    alpha_c = 2.063
    gamma_c = -135
    bright_contrast_img = cv.addWeighted(brightness_image, alpha_c, brightness_image, 0, gamma_c)
    
    # Convert image to grayscale
    gray_image = cv.cvtColor(bright_contrast_img, cv.COLOR_BGR2GRAY)

    # apply median blur to remove noise and preserves edges
    #   kernel size set to 9 (must be odd number)    
    filtered_image = cv.medianBlur(gray_image, 9)
    
    # Binary mask via Otsu to isolate the LEGO piece
    #   thresholding (convert to binary) - otsu method
    #   0 is an arbitrary threshold since otsu finds the optimal threshold
    #   Threshold type THRESH_BINARY_INV returns the lego piece in white - observed to be good for contour detection
    arbt_threshold = 0
    _, threshold_image = cv.threshold(filtered_image,arbt_threshold,255,cv.THRESH_BINARY_INV + cv.THRESH_OTSU)
    
    # Morphology to clean mask before moments
    kernel = np.ones((3, 3), np.uint8)
    smoothen_image = cv.morphologyEx(threshold_image, cv.MORPH_OPEN, kernel)
    smoothen_image = cv.morphologyEx(smoothen_image, cv.MORPH_CLOSE, kernel)
    
    # Find outer contour representing the piece silhouette
    contours, _ = cv.findContours(smoothen_image, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError('No contours found in image')
    
    # pick the most prominent contour
    cnt = max(contours, key=cv.contourArea)

    # Compute spatial moments then Hu invariants
    moments = cv.moments(cnt)
    # retrieve hu moments to represent the silhoutte of object
    hu_moments = cv.HuMoments(moments).flatten()
    # absolute value of all raw hu moment was taken to make it reflection/mirror invariant
    hu_moments = np.abs(hu_moments)
    # Log transform stabilizes dynamic range
    hu_moments = -np.log10(hu_moments)

    return hu_moments


In [6]:
# === Data loading helper ===

def load_image_data(folder_path, image_size, class_labels_dict):
    """
    Load LEGO images from a folder and assign class labels by filename prefix.

    This function scans a folder, filters files by known class prefixes, reads
    each image, resizes to a standard square size, and returns image arrays with
    corresponding labels.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing image files (training or testing set).
    image_size : int
        Target width/height in pixels for square resize.
    class_labels_dict : dict
        Mapping of class numbers to filename prefixes (e.g., {0: '2b1', 1: 'cir', ...}).

    Returns
    -------
    tuple
        x_data : numpy.ndarray
            Array of shape (n_samples, image_size, image_size, 3) containing RGB data.
        y_data : numpy.ndarray
            Array of shape (n_samples, 1) containing class labels.

    Notes
    -----
    - Only files whose names start with a known class prefix are loaded.
    - Raises if an image cannot be read to surface data issues early.
    """
    # List and sort files for deterministic ordering
    file_names = sorted(os.listdir(folder_path))
    matched = []
    # Match filenames to class prefixes; skip anything unexpected
    for fname in file_names:
        for class_number, label_prefix in class_labels_dict.items():
            if fname.startswith(str(label_prefix)):
                matched.append((fname, class_number))
                break  # stop at first match
    print(f"Matched files in {folder_path}: {len(matched)}")

    # Preallocate arrays sized to matched files
    x_data = np.empty((len(matched), image_size, image_size, 3), dtype=np.uint8)
    y_data = np.empty((len(matched), 1), dtype=np.int64)

    # Read, resize, and store each image with its label
    for idx, (filename, class_number) in enumerate(matched):
        image_path = os.path.join(folder_path, filename)
        image = cv.imread(image_path)
        if image is None:
            raise ValueError(f"Failed to read image: {image_path}")
        # Resize to uniform square so downstream features align
        image_array = cv.resize(image, (image_size, image_size))
        x_data[idx] = image_array.astype(np.uint8)
        y_data[idx, 0] = int(class_number)

    return x_data, y_data


In [7]:
# === Feature extraction orchestrator ===

def features_extraction(image):
    """
    Build a 15-dimensional feature vector for one LEGO image.

    This function aggregates stud count, circle detection, geometric measures,
    and Hu moments into a fixed-length vector for classification.

    Parameters
    ----------
    image : numpy.ndarray
        BGR image array representing a single LEGO piece.

    Returns
    -------
    numpy.ndarray
        Array of shape (1, 15) containing all engineered features.
    """
    # Preallocate feature row
    features = np.empty((1, 15))
    # Compute individual feature blocks from helper functions
    num_of_studs = process_stud_num(image)
    circ_detected = large_circle_detector(image)
    width, height, aspect_ratio, area, relative_area, perimeter = relative_sizes(image)
    hu_moments = process_hu_moments(image)
    # Populate feature vector consistently for train/test
    features[0][0] = num_of_studs
    features[0][1] = circ_detected
    features[0][2] = width
    features[0][3] = height
    features[0][4] = aspect_ratio
    features[0][5] = area
    features[0][6] = relative_area
    features[0][7] = perimeter
    features[0][8:] = hu_moments
    return features


In [8]:
# === Training pipeline ===

def training_function(path, image_size, class_labels):
    """
    Train the logistic regression model on LEGO images.

    This function loads the training dataset, extracts handcrafted features
    for each image, and fits a logistic regression classifier.

    Parameters
    ----------
    path : str
        Path to the training dataset folder.
    image_size : int
        Target width/height in pixels for square resize.
    class_labels : dict
        Mapping of class numbers to filename prefixes.

    Returns
    -------
    tuple
        x_train : numpy.ndarray
            Feature matrix of shape (n_samples, 15).
        y_train : numpy.ndarray
            Label array of shape (n_samples, 1).
    """
    # Load images and labels from disk
    x_imgs, y_train = load_image_data(path, image_size, class_labels)
    num_of_imgs = x_imgs.shape[0]
    # Preallocate feature matrix for all training images
    x_train = np.empty((num_of_imgs, 15))
    # Extract handcrafted features per image
    for i in range(num_of_imgs):
        #print(x_imgs[i])
        x_train[i, :] = features_extraction(x_imgs[i])
    return x_train, y_train

# Run training
x_train, y_train = training_function(FOLDER_TRAINING, IM_SIZE, CLASS_LABELS)
print(x_train.shape, y_train.shape)
# Fit logistic regression classifier on extracted features
model = LogisticRegression(max_iter= 50000, C = 1)
model.fit(x_train, y_train.ravel())


Matched files in Lego_dataset_2/training/: 108
(108, 15) (108, 1)


<local path>\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(C=1, max_iter=50000)

In [9]:
# === Training results (confusion matrix + accuracy) ===

# Predict on training set
train_preds = model.predict(x_train)
# Compute confusion matrix and accuracy
train_cm = confusion_matrix(y_train.ravel(), train_preds)
train_acc = accuracy_score(y_train.ravel(), train_preds)
# Display results
print('Training confusion matrix:')
print(train_cm)
print(f'Training accuracy: {train_acc:.4f}')


Training confusion matrix:
[[27  0  0  0]
 [ 1 25  0  1]
 [ 0  0 27  0]
 [ 1  1  0 25]]
Training accuracy: 0.9630


In [10]:
# === Testing pipeline ===

def testing_function(path, image_size, class_labels):
    """
    Test the trained model on a dataset and display performance metrics.

    This function loads test data, makes predictions, and evaluates the model's
    performance using confusion matrix and accuracy score.

    Parameters
    ----------
    path : str
        Path to the folder containing test images.
    image_size : int
        Width/height of processed square images.
    class_labels : dict
        Dictionary mapping class numbers to filename prefixes.

    Returns
    -------
    tuple
        x_test : numpy.ndarray
            Test feature matrix of shape (n_samples, 15).
        y_test : numpy.ndarray
            Label array of shape (n_samples, 1).
    """
    # Load images and labels from disk
    x_imgs, y_test = load_image_data(path, image_size, class_labels)
    num_of_imgs = x_imgs.shape[0]
    # Preallocate feature matrix for all test images
    x_test = np.empty((num_of_imgs, 15))
    # Extract same handcrafted features as training
    for i in range(num_of_imgs):
        x_test[i, :] = features_extraction(x_imgs[i])
    return x_test, y_test

# Prepare test features
x_test, y_test = testing_function(FOLDER_TESTING, IM_SIZE, CLASS_LABELS)


Matched files in Lego_dataset_2/testing/: 108


In [11]:
# === Testing results (confusion matrix + accuracy) ===

# Predict on test set
test_preds = model.predict(x_test)
# Compute confusion matrix and accuracy
test_cm = confusion_matrix(y_test.ravel(), test_preds)
test_acc = accuracy_score(y_test.ravel(), test_preds)
# Display results
print('Testing confusion matrix:')
print(test_cm)
print(f'Testing accuracy: {test_acc:.4f}')


Testing confusion matrix:
[[23  0  0  4]
 [ 3 24  0  0]
 [ 1  0 24  2]
 [ 0  0  0 27]]
Testing accuracy: 0.9074
